![image.png](attachment:image.png)

In [2]:
import torch
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

### Simplified Self Attention

![image.png](attachment:image.png)

In [4]:
# 1. Calculate Attention Scores
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, xi in enumerate(inputs):
    attn_scores_2[i] = torch.dot(query, xi)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [5]:
# 2. normalise the attention scores
attn_weight_2_temp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights: ", attn_weight_2_temp)
print("Sum of attention weights: ", attn_weight_2_temp.sum())

Attention weights:  tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum of attention weights:  tensor(1.0000)


![image.png](attachment:image.png)

In [6]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights (naive): ", attn_weights_2_naive)

print("Sum of attention weights (naive): ", attn_weights_2_naive.sum())

Attention weights (naive):  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum of attention weights (naive):  tensor(1.)


In [7]:
attn_weights_2 = torch.nn.functional.softmax(attn_scores_2, dim=0)
print("Attention weights (PyTorch): ", attn_weights_2)

Attention weights (PyTorch):  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


![image.png](attachment:image.png)

In [8]:
# 3. Obtaining the context vector
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [9]:
# computing self attention for all inputs
# Method-1 for loop
attn_scores = torch.empty(6,6)
for i, xi in enumerate(inputs):
    for j, xj in enumerate(inputs):
        attn_scores[i,j] = torch.dot(xi, xj)

print("Attention scores: ", attn_scores)

Attention scores:  tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [10]:
# Method 2 matrix multiplication
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [11]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print("Attention weights: ", attn_weights)

Attention weights:  tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [12]:
# check
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)
print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [13]:
all_context_vectors = attn_weights @ inputs
print("All context vectors: ", all_context_vectors)

All context vectors:  tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [14]:
# 3.4. Implementing self-attention with trainable weights

In [16]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [17]:
torch.manual_seed(123)
# initializing three weight matrices
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [18]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print("Query vector: ", query_2)
print("Key vector: ", key_2)
print("Value vector: ", value_2)

Query vector:  tensor([0.4306, 1.4551])
Key vector:  tensor([0.4433, 1.1419])
Value vector:  tensor([0.3951, 1.0037])


In [19]:
# weight parameters are the fundamental, learned coefficients that define
# the network’s connections, while attention weights are dynamic, context-specific values.

In [20]:
keys = inputs @ W_key
print("Keys: ", keys)
print("Keys shape: ", keys.shape)
value = inputs @ W_value
print("Values: ", value)
print("Values shape: ", value.shape)

Keys:  tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]])
Keys shape:  torch.Size([6, 2])
Values:  tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]])
Values shape:  torch.Size([6, 2])


In [21]:
# attention score of w22
keys_2 = keys[1]
attn_score_22 = torch.dot(query_2, keys_2)
print("Attention score of w22: ", attn_score_22)

Attention score of w22:  tensor(1.8524)


In [22]:
# attention scores
attn_scores_2 = query_2 @ keys.T
print("Attention scores: ", attn_scores_2)

Attention scores:  tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [23]:
# attention_weight = fn(attention_score) -> scale the attention scores to a range of 0 to 1, which can be interpreted as probabilities. 
# This is typically done using the softmax function, which exponentiates each score and normalizes them by dividing by the sum of all exponentiated scores. 
# The resulting attention weights indicate the relative importance of each input in relation to the query.    
d_k = keys.shape[-1]
attn_weights_2 = torch.nn.functional.softmax(attn_scores_2 / (d_k ** 0.5), dim=-1)
print("Attention weights: ", attn_weights_2)

Attention weights:  tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [24]:
context_vec_2 = attn_weights_2 @ value
print("Context vector: ", context_vec_2)

Context vector:  tensor([0.3061, 0.8210])


In [25]:
# A query is analogous to a search query in a database. It represents the current item
# (e.g., a word or token in a sentence) the model focuses on or tries to understand.
# The query is used to probe the other parts of the input sequence to determine how
# much attention to pay to them.

# The key is like a database key used for indexing and searching. In the attention mechanism,
# each item in the input sequence (e.g., each word in a sentence) has an associated
# key. These keys are used to match the query.

# The value in this context is similar to the value in a key-value pair in a database. It
# represents the actual content or representation of the input items. Once the model
# determines which keys (and thus which parts of the input) are most relevant to the
# query (the current focus item), it retrieves the corresponding values.

In [26]:
# attention_score = query @ key
# attention_weight = torch.nn.functional.softmax(attention_score, dim=-1)
# context_vector = attention_weight @ value

In [27]:
import torch.nn as nn
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
    
    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / (keys.shape[-1] ** 0.5), dim=-1
        )
        context_vector = attn_weights @ values
        return context_vector
    


In [28]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in=d_in, d_out=d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [29]:
# using linear layers => because it has an optimized weight initialization, contributing to more stable and effective model training.
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=True):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / (keys.shape[-1] ** 0.5), dim=-1
        )
        context_vector = attn_weights @ values
        return context_vector

In [30]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.3427, -0.2720],
        [-0.3439, -0.2692],
        [-0.3438, -0.2693],
        [-0.3417, -0.2728],
        [-0.3412, -0.2736],
        [-0.3426, -0.2714]], grad_fn=<MmBackward0>)
